# 📘 Notebook 2e: Orchestrating ML Jobs with Airflow

## Overview

You've seen the model train interactively in `02_GPU_Training_Wafers`. In production you don't click "Run" every morning — your orchestrator does. This notebook answers the customer's natural follow-up:

> *"How does this connect to our Airflow stack?"*

The short answer: **same DAG, fewer vendors.** Airflow stays in charge; Snowflake ML Jobs replace the SageMaker / EKS / custom-compute step inside the task.

### What's Covered

| Section | Topic |
|---|---|
| 1 | What an ML Job is + why use it |
| 2 | Two ways to kick one off — `@remote` decorator and `submit_file` |
| 3 | The wafer model as an ML Job (production-ready) |
| 4 | Orchestrating from Airflow — same DAG, different operator |
| 5 | Monitoring, logging, and what happens when a job fails |
| 6 | Summary |

### References

- [Snowflake ML Jobs docs](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview)
- [Snowflake-Labs/sf-samples — `ml_jobs/`](https://github.com/Snowflake-Labs/sf-samples/tree/main/samples/ml/ml_jobs) — the `xgb_classifier_airflow` folder is the canonical Airflow + ML Jobs reference
- [apache-airflow-providers-snowflake](https://airflow.apache.org/docs/apache-airflow-providers-snowflake/stable/)


## 📘 Section 1 — What is an ML Job

An ML Job is a **Python script or function that runs on a SPCS compute pool** — the same container runtime that powers Snowflake Notebooks — submitted from outside.

```
Your code (anywhere)  ──►  submit_file("train.py", compute_pool=...)  ──►  runs in Snowflake
```

### Why use one

| Benefit | Why it matters for this demo |
|---|---|
| **Same compute as notebooks** | If it runs in `02_GPU_Training_Wafers`, it runs as an ML Job — no re-engineering |
| **Triggerable from anywhere** | Airflow, dbt, cron, CI/CD, a laptop — all via a single Python call |
| **No infra to own** | No Docker image to build, no cluster to provision, no kubectl |
| **Data stays in Snowflake** | Reads tables and feature views directly — no S3/GCS staging |
| **Standard governance** | Same RBAC, same billing, same event table as anything else in the account |

### The alternative it replaces

Most customers today do this with Airflow + SageMaker (or Airflow + EKS). That means: unload features to S3 → spin up a SageMaker training container → artifacts back to S3 → SageMaker Model Registry → IAM + Snowflake RBAC (two security models). With ML Jobs the orchestration layer (Airflow) stays identical; the compute layer moves next to the data.


## 📘 Section 2 — Two ways to kick off an ML Job

From [the sf-samples README](https://github.com/Snowflake-Labs/sf-samples/blob/main/samples/ml/ml_jobs/README.md), there are two canonical entry points — both in `snowflake.ml.jobs`:

| API | When to use |
|---|---|
| `@remote` decorator | Quick dispatch of a Python function. Great for notebooks, one-offs, and keeping the DAG authoring experience tight. |
| `submit_file` / `submit_directory` / `submit_from_stage` | When your training lives as a `.py` file in your git repo (production pattern). |

Both return an `MLJob` handle with `.id`, `.status`, `.wait()`, `.get_logs()`, `.cancel()`.

The cell below is a hello-world using `@remote` — finishes in seconds, proves the path works.


In [ ]:
# ============================================================================
# HELLO WORLD — @remote decorator
# ============================================================================
from snowflake.snowpark.context import get_active_session
from snowflake.ml.jobs import remote

session = get_active_session()
session.sql("USE DATABASE WAFER_YIELD_DEMO").collect()
session.sql("USE SCHEMA ML_MODELS").collect()

COMPUTE_POOL = "WAFER_INFERENCE_POOL"          # small CPU pool — fast for hello-world
JOB_STAGE    = "@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE"

@remote(COMPUTE_POOL, stage_name=JOB_STAGE)
def hello_from_spcs(name: str = "world"):
    # Import inside the function — the body is serialized and run remotely
    import os, sys
    from datetime import datetime
    print(f"{datetime.now()}  Hello {name} from Python {sys.version_info[:2]}")
    print(f"Env var RUN_ID = {os.environ.get('RUN_ID', '<unset>')}")

job = hello_from_spcs("customer demo")
print(f"Job ID: {job.id}")
print(f"Initial status: {job.status}")
job.wait()
print(f"Final status: {job.status}")
print("--- logs ---")
print(job.get_logs())


## 📘 Section 3 — The wafer model as an ML Job

This is the production-ready version of the DNN training from `02_GPU_Training_Wafers`. Three things are different from the notebook version:

1. The whole thing is one file (or one `@remote` function) — so it can live in a git repo and be submitted on a schedule
2. It reads the dataset *version* from an env var so Airflow can point it at fresh data each run
3. It prints a JSON summary as the last line of stdout so the orchestrator can capture the new model version via XCom

The cell below defines the training as a `@remote` function — fastest way to see it work. In Section 4 we'll show the same thing as a standalone file that Airflow references.


In [ ]:
# ============================================================================
# WAFER MODEL TRAINING AS AN ML JOB
# ============================================================================
from datetime import datetime
from snowflake.ml.jobs import remote

NEW_VERSION = f"V_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

@remote(COMPUTE_POOL, stage_name=JOB_STAGE, env_vars={"NEW_VERSION_NAME": NEW_VERSION})
def train_wafer_model(train_version: str = "v1_train", test_version: str = "v1_test"):
    import os, json
    import torch
    import torch.nn as nn
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml import dataset
    from snowflake.ml.data import DataConnector
    from snowflake.ml.registry import Registry

    session = get_active_session()
    DATASET = "WAFER_YIELD_DEMO.RAW_DATA.WAFER_YIELD_TRAINING_DATASET"
    MODEL_NAME = "WAFER_YIELD_CHAMPION"
    version_name = os.environ["NEW_VERSION_NAME"]

    # Load train + test splits — both live in Snowflake, no data movement
    train = DataConnector.from_dataset(dataset.load_dataset(session, DATASET, train_version)).to_pandas()
    test  = DataConnector.from_dataset(dataset.load_dataset(session, DATASET, test_version)).to_pandas()

    EXCLUDE = ['WAFER_ID','FIRST_TIMESTAMP','YIELD_GOOD','YIELD_SCORE','DOMINANT_DEFECT_TYPE']
    feats = [c for c in train.columns if c.upper() not in EXCLUDE]

    # Train a small DNN (same architecture as Notebook 02's baseline)
    Xtr = torch.tensor(train[feats].astype('float32').values)
    ytr = torch.tensor(train['YIELD_GOOD'].astype('float32').values).unsqueeze(1)
    model = nn.Sequential(
        nn.Linear(len(feats), 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32, 1), nn.Sigmoid(),
    )
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.BCELoss()
    model.train()
    for epoch in range(20):
        opt.zero_grad(); loss = loss_fn(model(Xtr), ytr); loss.backward(); opt.step()

    # Held-out test accuracy
    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(test[feats].astype('float32').values)).squeeze().numpy()
    acc = float(((probs >= 0.5).astype(int) == test['YIELD_GOOD'].values).mean())

    # Register and print a machine-parseable summary for the orchestrator
    reg = Registry(session, database_name="WAFER_YIELD_DEMO", schema_name="ML_MODELS")
    reg.log_model(
        model=model, model_name=MODEL_NAME, version_name=version_name,
        comment=f"ML Job training against {train_version}",
        metrics={"accuracy": acc},
        sample_input_data=train[feats].head(1).astype('float32'),
    )
    print(json.dumps({"version": version_name, "accuracy": acc}))


# Invoke — this submits the job, does NOT run it locally
job = train_wafer_model(train_version="v1_train", test_version="v1_test")
print(f"Submitted job: {job.id}  (target version: {NEW_VERSION})")
print("Waiting for completion...")
job.wait()
print(f"Final status: {job.status}")
print("--- tail of logs ---")
print(job.get_logs().splitlines()[-5:])


## 📘 Section 3b — Developing ML Jobs locally (VS Code)

The same `@remote` / `submit_file` API works from your laptop. You write Python in VS Code, hit submit, and the job runs in Snowflake on SPCS — you never touch a Docker image or a cluster.

### Why this matters for the customer

- Their data scientists already live in VS Code / PyCharm / Cursor
- They keep their git workflow, linter, debugger, and unit tests
- They only pay for GPU time when they actually submit — local iteration is free
- The exact same code that runs locally runs in Airflow — no rewrite when productionizing

### Minimal laptop setup

```bash
# 1. Python 3.10 venv (ML Jobs currently requires 3.10)
python3.10 -m venv .venv && source .venv/bin/activate

# 2. Install the client
pip install "snowflake-ml-python>=1.9.0"

# 3. Configure a connection (one-time)
#    ~/.snowflake/connections.toml
cat >> ~/.snowflake/connections.toml <<'EOF'
[haleyconnect]
account = "sfsenorthamerica-hmassa_aws1"
user = "HMASSA"
authenticator = "SNOWFLAKE_JWT"
private_key_file = "/Users/hmassa/.snowflake/keys/rsa_key.p8"
role = "ACCOUNTADMIN"
warehouse = "WAFER_DEMO_WH"
database = "WAFER_YIELD_DEMO"
schema = "ML_MODELS"
EOF

# 4. (Optional) Install the Snowflake VS Code extension
#    https://marketplace.visualstudio.com/items?itemName=Snowflake.snowflake-vsc
```

### Recommended repo layout

```
wafer-ml/
├── ml/
│   ├── train_wafer_model.py      # the ML Job entry point (runs on SPCS)
│   └── tests/
│       └── test_features.py      # local unit tests — run with pytest before submitting
├── dags/
│   └── wafer_yield_pipeline.py   # Airflow DAG that calls submit_file(...)
├── submit_training.py            # dev helper — run this from VS Code to submit a test job
└── pyproject.toml
```

### Dev loop

1. Edit `ml/train_wafer_model.py` — full VS Code IntelliSense, type-check, local debugger
2. Run local unit tests: `pytest ml/tests/` — pure-Python bits (feature transforms, parsing) run instantly
3. When you need the full GPU run: `python submit_training.py` → ships the file to SPCS, streams logs back
4. Once it passes in the cloud, commit to git. Airflow picks up `ml/train_wafer_model.py` on its next run.

The cell below is what `submit_training.py` looks like — it's the single file a developer runs from VS Code.


In [ ]:
# ============================================================================
# submit_training.py — run this from VS Code to submit a dev job
# ============================================================================
# This is the "laptop → Snowflake" entry point. Exact same API that Airflow uses
# in production, just invoked by hand for faster iteration.
from datetime import datetime
from snowflake.snowpark import Session
from snowflake.ml.jobs import submit_file

# Pick up the named connection from ~/.snowflake/connections.toml
session = Session.builder.config("connection_name", "haleyconnect").create()

version = f"V_DEV_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
job = submit_file(
    "ml/train_wafer_model.py",
    compute_pool="WAFER_TRAINING_POOL",
    stage_name="@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE",
    env_vars={
        "DATASET_VERSION":      "v1_train",
        "DATASET_TEST_VERSION": "v1_test",
        "NEW_VERSION_NAME":     version,
    },
)

print(f"Submitted: {job.id}")
print(f"Version:   {version}")
print("Streaming logs until job completes...")
job.wait()
print(f"Final: {job.status}")
print(job.get_logs())


## 📘 Section 4 — Orchestrating with Airflow

The customer's Airflow stack doesn't change. They swap whatever SageMaker / EKS / shell operator they're using today for a `SnowparkOperator` that calls `submit_file(...)`.

### Same DAG shape, fewer vendors

```
BEFORE: Unload to S3 → SageMaker Training → SageMaker Model Registry → SageMaker Transform → Load back to Snowflake
AFTER:  Snowpark ML Job (train + register) → SQL (alias promote) → SQL (batch score)
```

5 tasks → 3 tasks. Two auth domains (IAM + Snowflake) → one. No S3 round-trip.

The DAG file below is drop-in for a customer's `dags/` folder. We don't execute Airflow from this notebook — we just print it.


In [ ]:
# ============================================================================
# dags/wafer_yield_pipeline.py  (illustrative — lives in the customer's Airflow repo)
# ============================================================================
DAG_SOURCE = 'from datetime import datetime\nfrom airflow import DAG\nfrom airflow.providers.snowflake.operators.snowflake import SnowflakeSqlApiOperator\nfrom airflow.providers.snowflake.operators.snowpark import SnowparkOperator\n\nSNOWFLAKE_CONN_ID = "snowflake_prod"\nMIN_ACCURACY_TO_PROMOTE = 0.70\n\n\ndef submit_training_job(**context):\n    """Runs inside SnowparkOperator (authenticated session available)."""\n    import json\n    from datetime import datetime\n    from snowflake.ml.jobs import submit_file\n\n    new_version = f"V_{datetime.utcnow().strftime(\'%Y%m%d_%H%M%S\')}"\n    job = submit_file(\n        "/opt/airflow/dags/ml/train_wafer_model.py",      # lives in the customer\'s repo\n        compute_pool="WAFER_TRAINING_POOL",\n        stage_name="@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE",\n        env_vars={\n            "DATASET_VERSION": context["ds_nodash"],\n            "DATASET_TEST_VERSION": "v1_test",\n            "NEW_VERSION_NAME": new_version,\n        },\n    )\n    job.wait()\n    logs = job.get_logs()\n    if job.status != "DONE":\n        # ── FAILURE HANDLING ── bubble the container logs into the Airflow task log\n        raise RuntimeError(f"ML Job failed (status={job.status})\\n---\\n{logs}")\n\n    # Last JSON line of the training script is {"version": ..., "accuracy": ...}\n    summary_line = [ln for ln in logs.splitlines() if ln.strip().startswith("{")][-1]\n    return json.loads(summary_line)                       # becomes XCom\n\n\nwith DAG(\n    "wafer_yield_pipeline",\n    start_date=datetime(2026, 1, 1),\n    schedule="0 2 * * *",\n    catchup=False,\n    default_args={"retries": 2, "retry_delay": 300},      # Airflow-level retries\n) as dag:\n\n    train = SnowparkOperator(\n        task_id="train",\n        snowflake_conn_id=SNOWFLAKE_CONN_ID,\n        python_callable=submit_training_job,\n    )\n\n    # Promote ONLY if the new model clears the accuracy gate — quality gate in the DAG\n    promote = SnowflakeSqlApiOperator(\n        task_id="promote_to_prod",\n        snowflake_conn_id=SNOWFLAKE_CONN_ID,\n        sql=(\n            "DECLARE "\n            "  v_ver STRING DEFAULT \'{{ ti.xcom_pull(task_ids=\\"train\\").version }}\'; "\n            "  v_acc FLOAT  DEFAULT {{ ti.xcom_pull(task_ids=\\"train\\").accuracy }}; "\n            "BEGIN "\n            f"  IF (v_acc >= {MIN_ACCURACY_TO_PROMOTE}) THEN "\n            "    ALTER MODEL WAFER_YIELD_DEMO.ML_MODELS.WAFER_YIELD_CHAMPION "\n            "      SET ALIAS PROD = :v_ver; "\n            "    RETURN \'promoted \' || :v_ver; "\n            "  ELSE "\n            "    RETURN \'not promoted - accuracy \' || :v_acc || \' below gate\'; "\n            "  END IF; "\n            "END;"\n        ),\n    )\n\n    score = SnowflakeSqlApiOperator(\n        task_id="daily_batch_score",\n        snowflake_conn_id=SNOWFLAKE_CONN_ID,\n        sql=(\n            "INSERT INTO WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS "\n            "SELECT WAFER_ID, "\n            "  WAFER_YIELD_DEMO.ML_MODELS.WAFER_YIELD_CHAMPION!FORWARD(*):output_feature_0 AS YIELD_PROB, "\n            "  CURRENT_TIMESTAMP() AS SCORED_AT, "\n            "  \'{{ ds }}\' AS RUN_DATE "\n            "FROM WAFER_YIELD_DEMO.INFERENCE.WAFER_INFERENCE_INPUT "\n            "WHERE INGESTION_DATE = \'{{ ds }}\';"\n        ),\n    )\n\n    train >> promote >> score\n'
print(DAG_SOURCE)


### Connection setup (one-time on the Airflow side)

One Snowflake connection — everything else (train, promote, score) uses it.

```bash
airflow connections add snowflake_prod \
    --conn-type snowflake \
    --conn-host <account>.snowflakecomputing.com \
    --conn-login <service_user> \
    --conn-schema ML_MODELS \
    --conn-extra '{"database":"WAFER_YIELD_DEMO","warehouse":"WAFER_DEMO_WH","role":"ML_SERVICE_ROLE","private_key_file":"/opt/airflow/keys/sf_service.p8"}'
```

Best practice: **service user + key-pair auth**, not password. The provider supports it natively.


## 📘 Section 5 — Monitoring, Logging, and What Happens When a Job Fails

Three separate surfaces, three different audiences:

| Surface | Who looks at it | What it answers |
|---|---|---|
| **Airflow UI** | On-call engineer | Did the DAG run, which task failed, should I retry |
| **ML Job logs** (`job.get_logs()`) | ML engineer | What did the training script print, did it OOM |
| **Event Table** (`SNOWFLAKE.TELEMETRY.EVENTS`) | Platform / FinOps | Historical, structured, queryable, long-lived |

### Failure modes and how they're handled

| Failure | What Airflow sees | How to recover |
|---|---|---|
| **ML Job container OOM** | `job.status == "FAILED"`, task raises | Airflow auto-retry (see `retries=2`); increase compute pool size if persistent |
| **Compute pool cold / suspended** | Slow start on first task | `ALTER COMPUTE POOL ... RESUME` as a DAG pre-task, or set `auto_resume = TRUE` |
| **Training accuracy below gate** | Job succeeds, `promote` SQL no-ops | Alert via Airflow callback on the `promote` XCom return value |
| **Snowflake auth failure** | `SnowparkOperator` raises up front | Fix connection / key rotation |
| **Script bug / exception** | Job status `FAILED`, logs show traceback | `job.get_logs()` surfaces full stack trace — printed to Airflow task log by the `RuntimeError` above |

The failure-handling in the DAG above already does the important part: if `job.status != "DONE"`, it raises with the container logs attached, so Airflow's UI shows the Python traceback from inside the job — not just "task failed."


In [ ]:
# ============================================================================
# LIVE INSPECTION — status, logs, compute pool, recent jobs in the account
# ============================================================================
# This is what your DAG's failure path (or a debug notebook) would query.
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# 1. Most recent ML Job submitted earlier in this notebook
print(f"Last training job:      {job.id}")
print(f"Final status:           {job.status}")
print("--- last 10 log lines ---")
for ln in job.get_logs().splitlines()[-10:]:
    print(ln)

# 2. All ML Jobs in the account (they surface as SPCS services with managing_object_domain='Job')
print("\n--- recent services (filtered to this schema) ---")
services = session.sql(
    "SHOW SERVICES IN SCHEMA WAFER_YIELD_DEMO.ML_MODELS"
).to_pandas()
print(services[['name','status','compute_pool','created_on']].head(10))

# 3. Compute pool state
print("\n--- compute pool state ---")
pool = session.sql("DESCRIBE COMPUTE POOL WAFER_TRAINING_POOL").to_pandas()
print(pool[['name','state','min_nodes','max_nodes','instance_family']].iloc[0])


In [ ]:
# ============================================================================
# STRUCTURED LOGS VIA THE EVENT TABLE
# ============================================================================
# Enable once per account:
#   ALTER ACCOUNT SET EVENT_TABLE = <db>.<schema>.<event_table>;
# After that every ML Job's stdout/stderr, spans, and metrics land here and can
# be joined to other Snowflake audit data.
try:
    events = session.sql(
        '''
        SELECT TIMESTAMP,
               RESOURCE_ATTRIBUTES:"snow.service.name"::STRING AS service,
               RECORD_TYPE,
               VALUE
        FROM SNOWFLAKE.TELEMETRY.EVENTS
        WHERE RESOURCE_ATTRIBUTES:"snow.service.name"::STRING ILIKE '%WAFER%'
        ORDER BY TIMESTAMP DESC
        LIMIT 20
        '''
    ).to_pandas()
    print(events)
except Exception as e:
    print(f"Event table not configured yet: {e}")
    print("Enable with: ALTER ACCOUNT SET EVENT_TABLE = <db>.<schema>.<event_table>;")


## 📘 Section 6 — Summary

### The one-sentence answer for the customer

> **Keep Airflow. Every training step is a `submit_file()` (or `@remote`) to an ML Job. Every scoring / promotion step is a SQL operator. The data never leaves Snowflake, and failures surface in the Airflow UI with full container logs attached.**

### What this notebook demonstrated

| Section | Takeaway |
|---|---|
| 1 | ML Jobs run the same container-runtime compute as your notebooks, but are submittable from anywhere |
| 2 | Two entry points: `@remote` (function) and `submit_file` (repo-tracked script) |
| 3 | The wafer champion training, rewritten as a production ML Job — prints a JSON summary for XCom |
| 4 | Drop-in Airflow DAG: train (SnowparkOperator) → promote (SQL, quality-gated) → score (SQL). Same DAG shape, two fewer vendors. |
| 5 | Three observability surfaces (Airflow UI, `job.get_logs()`, Event Table) and explicit failure-mode handling |

### Related references

- [sf-samples `ml_jobs/xgb_classifier_airflow`](https://github.com/Snowflake-Labs/sf-samples/tree/main/samples/ml/ml_jobs/xgb_classifier_airflow) — canonical Airflow + ML Jobs example
- [sf-samples `ml_jobs/e2e_task_graph`](https://github.com/Snowflake-Labs/sf-samples/tree/main/samples/ml/ml_jobs/e2e_task_graph) — if they don't have Airflow: same pattern with Snowflake's native Task Graph
- [ML Job API reference](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview)
